In [1]:
import importlib


class EmbeddedAIbot:

    def __init__(
        self,
        vector_db_path="vector_db",
        collection_name="workshop_rag_v2",
        embedding_model="sentence-transformers/all-MiniLM-L6-v2",
        llm_model="llama3.2:latest",
        top_k=5,
        candidate_k=20,
        max_context_documents=5,
        temperature=0.3,
        num_ctx=2048
    ):

        print("Initializing EmbeddedAIbot...\n")

        from src.processing.embedder import WorkshopEmbedder

        self.embedder = WorkshopEmbedder(
            model_name=embedding_model
        )

        print("✓ Embedding model loaded")

        from src.processing.vector_store import WorkshopVectorStore

        self.vector_store = WorkshopVectorStore(
            embedding_function=self.embedder.embeddings,
            persist_directory=vector_db_path,
            collection_name=collection_name
        )

        print("✓ Vector database connected")

        import src.rag.retriever
        importlib.reload(src.rag.retriever)

        from src.rag.retriever import WorkshopRetriever

        self.retriever = WorkshopRetriever(
            vector_store=self.vector_store,
            k=top_k,
            candidate_k=candidate_k
        )

        print("✓ Retriever loaded")


        import src.rag.context_builder
        importlib.reload(src.rag.context_builder)

        from src.rag.context_builder import ContextBuilder

        self.context_builder = ContextBuilder(
            max_documents=max_context_documents
        )

        print("✓ Context builder loaded")


        import src.rag.prompt
        importlib.reload(src.rag.prompt)

        from src.rag.prompt import WorkshopPrompt

        self.prompt_builder = WorkshopPrompt()

        print("✓ Prompt builder loaded")

        import src.rag.response_builder
        importlib.reload(src.rag.response_builder)

        from src.rag.response_builder import ResponseBuilder

        self.response_builder = ResponseBuilder()

        print("✓ Response builder loaded")


        from src.rag.llm import WorkshopLLM

        self.llm = WorkshopLLM(
            model_name=llm_model,
            temperature=temperature,
            num_ctx=num_ctx
        )

        print("✓ Ollama LLM connected")

        import src.rag.rag_chain
        importlib.reload(src.rag.rag_chain)

        from src.rag.rag_chain import WorkshopRAGChain

        self.rag_chain = WorkshopRAGChain(
            retriever=self.retriever,
            context_builder=self.context_builder,
            prompt_builder=self.prompt_builder,
            response_builder=self.response_builder,
            llm=self.llm,
            k=top_k
        )

        print("✓ RAG chain connected")

        # print("\n" + "=" * 50)
        print("EmbeddedAIbot READY")
        # print("=" * 50)


    def ask(self, query):

        if not query or not query.strip():
            return {
                "answer": "Please enter a question.",
                "citations": []
            }

        return self.rag_chain.invoke(query)

    def display(self, response):

        print("\nANSWER")
        # print("-" * 50)

        print(response["answer"])


        print("\nSOURCES")
        # print("-" * 50)

        for i, citation in enumerate(
            response["citations"],
            start=1
        ):

            print(
                f"[{i}] "
                f"{citation['file_name']} | "
                f"Slide/Page {citation['slide_number']} | "
                f"Type: {citation['content_type']}"
            )

In [2]:
bot = EmbeddedAIbot()

Initializing EmbeddedAIbot...



C:\Users\jashw\miniconda3\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model loaded
✓ Vector database connected
✓ Retriever loaded
✓ Context builder loaded
✓ Prompt builder loaded
✓ Response builder loaded
✓ Ollama LLM connected
✓ RAG chain connected
EmbeddedAIbot READY


In [6]:
query = input("You: ")

response = bot.ask(query)

bot.display(response)


You:  Explain the firmware architecture used in workshop , including the role of BSP , HAL , drivers , middleware/services , and the application layer



ANSWER
According to the workshop material, the firmware architecture consists of a layered stack with the following components:

1. **Application Layer**: This layer contains the product features and business logic. It is the topmost layer of the firmware stack and is responsible for implementing the product's functionality.
2. **Middleware/Services Layer**: This layer provides reusable cross-cutting services shared by the application. It includes components such as the RTOS scheduler, TCP/IP stack, filesystem, and logging framework.
3. **Drivers Layer**: This layer contains peripheral- or device-specific logic built on top of the HAL. Drivers issue I²C reads or other device-specific operations through the HAL layer.
4. **Hardware Abstraction Layer (HAL) Layer**: This layer provides a uniform API over peripheral registers, hiding the register-level detail of a peripheral family. The HAL layer acts as an intermediary between the application and the hardware.
5. **Board Support Package 